In [2]:
from collections import defaultdict
from pathlib import Path

import jsonlines
import json
import re

### Were the questions interrupted during the cleaning process?

— No

In [29]:
with jsonlines.open(f"original/train.jsonl", "r") as f:
    train_original = [entry for entry in f]

with jsonlines.open(f"cleaned/train_question.jsonl", "r") as f:
    train_cleaned = [entry for entry in f]

In [39]:
with jsonlines.open(f"cleaned/train_till_50712.jsonl", "r") as f:
    train_cleaned_unfinished = [entry for entry in f]
len(train_cleaned_unfinished)

50748

In [22]:
print(json.dumps(train_original[0], indent=4))

{
    "question": "Chronic urethral obstruction due to benign prismatic hyperplasia can lead to the following change in kidney parenchyma",
    "exp": "Chronic urethral obstruction because of urinary calculi, prostatic hyperophy, tumors, normal pregnancy, tumors, uterine prolapse or functional disorders cause hydronephrosis which by definition is used to describe dilatation of renal pelvis and calculus associated with progressive atrophy of the kidney due to obstruction to the outflow of urine Refer Robbins 7yh/9,1012,9/e. P950",
    "cop": 3,
    "opa": "Hyperplasia",
    "opb": "Hyperophy",
    "opc": "Atrophy",
    "opd": "Dyplasia",
    "subject_name": "Anatomy",
    "topic_name": "Urinary tract",
    "id": "e9ad821a-c438-4965-9f77-760819dfa155",
    "choice_type": "single"
}


In [4]:
first_run_last_q = "Mi's expression of the following homeobox genes alters the position of the forelimbs during development"
first_run_last_id = 50712
second_run_first_q = "CSF sample is preserved for which poisoning: FMGE 10"
second_run_first_id = 50748

In [25]:
missing_questions = []

In [33]:
len(train_original), len(train_cleaned)

(182822, 62702)

In [37]:
for i, (original, cleaned) in enumerate(zip(train_original, train_cleaned)):
    if original["question"] != cleaned["question"]:
        print("Mismatch at index", i, "original id", original["id"], "cleaned id", cleaned["id"])
    elif first_run_last_id < i < second_run_first_id:
        print(f"At index {i} questions are the same")

At index 50713 questions are the same
At index 50714 questions are the same
At index 50715 questions are the same
At index 50716 questions are the same
At index 50717 questions are the same
At index 50718 questions are the same
At index 50719 questions are the same
At index 50720 questions are the same
At index 50721 questions are the same
At index 50722 questions are the same
At index 50723 questions are the same
At index 50724 questions are the same
At index 50725 questions are the same
At index 50726 questions are the same
At index 50727 questions are the same
At index 50728 questions are the same
At index 50729 questions are the same
At index 50730 questions are the same
At index 50731 questions are the same
At index 50732 questions are the same
At index 50733 questions are the same
At index 50734 questions are the same
At index 50735 questions are the same
At index 50736 questions are the same
At index 50737 questions are the same
At index 50738 questions are the same
At index 507

In [43]:
for i, entry in enumerate(train_cleaned):
    print(f'- {entry["exp"]}')
    if i == 10:
        break

- Chronic urethral obstruction because of urinary calculi, prostatic hyperophy, tumors, normal pregnancy, tumors, uterine prolapse or functional disorders cause hydronephrosis which by definition is used to describe dilatation of renal pelvis and calculus associated with progressive atrophy of the kidney due to obstruction to the outflow of urine Refer Robbins 7yh/9,1012,9/e. P950
- Ans. (c) Vitamin B12 Ref: Harrison's 19th ed. P 640* Vitamin B12 (Cobalamin) is synthesized solely by microorganisms.* In humans, the only source for humans is food of animal origin, e.g., meat, fish, and dairy products.* Vegetables, fruits, and other foods of nonanimal origin doesn't contain Vitamin B12 .* Daily requirements of vitamin Bp is about 1-3 pg. Body stores are of the order of 2-3 mg, sufficient for 3-4 years if supplies are completely cut off.
- Ans. is 'd' i.e., Roux en Y Duodenal Bypass Bariatric surgical procedures include:a. Vertical banded gastroplastyb. Adjustable gastric bandingc. Roux-en

### Standardize the entries (not done yet)

In [40]:
with jsonlines.open(f"cleaned/train_question.jsonl", "r") as f:
    train_cleaned = [entry for entry in f]

with jsonlines.open(f"cleaned/dev_question.jsonl", "r") as f:
    dev_cleaned = [entry for entry in f]

with jsonlines.open(f"cleaned/test_question.jsonl", "r") as f:
    test_cleaned = [entry for entry in f]

In [41]:
for split in [train_cleaned, dev_cleaned, test_cleaned]:
    for i, entry in enumerate(split):
        if "comment_in_question_upd" not in entry:
            split[i]["comment_in_question_upd"] = entry.pop("comment_present")

In [ ]:
with jsonlines.open(f"cleaned/train_question.jsonl", "w", flush=True) as f:
    [f.write(entry) for entry in train_cleaned]

with jsonlines.open(f"cleaned/dev_question.jsonl", "w", flush=True) as f:
    [f.write(entry) for entry in dev_cleaned]

with jsonlines.open(f"cleaned/test_question.jsonl", "w", flush=True) as f:
    [f.write(entry) for entry in test_cleaned]

### Fix Weird Formatting

In [13]:
def remove_weird_spaces(split: str) -> tuple[int, dict[str, int]]:
    """
    Replace weird spaces with regular spaces in the question field of the entries in the given split and save the cleaned entries to a new file.
    :param split: The split to process (e.g., "train", "dev", "test")
    :return: A tuple containing the number of contaminated fields and a dictionary with the count of contaminated fields per key
    """
    num_contaminated_fields = 0
    contaminated_fields = defaultdict(int)
    weird_space = " "
    with jsonlines.open(f"original/{split}.jsonl", "r") as f:
        entries = [entry for entry in f]

    updated_entries = []
    for i, entry in enumerate(entries):
        for key, value in entry.items():
            if isinstance(value, str) and weird_space in value:
                entry[key] = value.replace(weird_space, " ")
                num_contaminated_fields += 1
                contaminated_fields[key] += 1
            else:
                entry[key] = value
        updated_entries.append(entry)

    assert len(updated_entries) == len(entries), \
        "The number of entries should remain the same after cleaning."

    with jsonlines.open(f"original/clean_spaces/{split}.jsonl", "w", flush=True) as f:
        [f.write(entry) for entry in entries]

    return num_contaminated_fields, contaminated_fields

In [14]:
for split in ["train", "dev", "test"]:
    print(f"Split: {split}")
    num, stats = remove_weird_spaces(split)
    print(f"Number of contaminated fields: {num}")
    print("Contaminated fields per key:")
    for key, count in stats.items():
        print(f" - {key}: {count}")

Split: train
Number of contaminated fields: 6662
Contaminated fields per key:
 - exp: 6602
 - question: 44
 - opb: 3
 - opa: 5
 - opd: 6
 - opc: 2
Split: dev
Number of contaminated fields: 245
Contaminated fields per key:
 - exp: 243
 - question: 1
 - opc: 1
Split: test
Number of contaminated fields: 5
Contaminated fields per key:
 - question: 4
 - opb: 1


### Duplicated Items?

In [51]:
with jsonlines.open(f"cleaned/train_explanation.jsonl", "r") as f:
    entries = [entry for entry in f]
    grouped_by_question = defaultdict(list)
    for entry in entries:
        grouped_by_question[entry["question"]].append(entry)
    if any(len(group) > 1 for group in grouped_by_question.values()):
        print("There are duplicated questions in the cleaned train set.")
    else:
        print("There are no duplicated questions in the cleaned train set.")

There are no duplicated questions in the cleaned train set.


### Fixing Variables

In [10]:
updated = []
with jsonlines.open("cleaned/dev_explanation_backup.jsonl", "r") as f:
    for i, entry in enumerate(f):
        if not entry["exp"]:
            entry["exp_to_edit"] = None
            entry["exp_upd"] = None
        else:
            if "exp_to_edit" in entry:
                if entry["exp_to_edit"] is False:
                    if "exp_upd" in entry:
                        entry.pop("exp_upd")
            else:
                if "exp_upd" in entry:
                    entry["exp_to_edit"] = None

        updated.append(entry)

In [12]:
len(updated)

5101

In [11]:
out_path = Path("cleaned/dev_explanation.jsonl")
with jsonlines.open(out_path, "w", flush=True) as f:
    [f.write(entry) for entry in updated]

assert out_path.exists(), "Output file was not created successfully."
print(f"Updated train set saved to {out_path}")

Updated train set saved to cleaned/train_explanation.jsonl


### Fixing questions (restoring from the log)

In [16]:
with jsonlines.open(f"cleaned/train_question.jsonl", "r") as f:
    entries = [entry for entry in f] # locally and on cluster: 62702; in log: 92554
    print(f"Number of contaminated fields: {len(entries)}")

Number of contaminated fields: 62702


In [28]:
with open(f"cleaned/cleaning_q_out_train_64029_92554", "r") as f:
    log = f.read()
    split_pattern = re.compile(r"\n+\[train\] Entry \d+:\n+")
    items = split_pattern.split(log)[1:]  # Skip the first empty split
    items = [item.strip().split("\n") for item in items if item.strip()]  # Remove empty items and strip whitespace
    print(f"Number of items in log: {len(items)}")

Number of items in log: 28526


In [29]:
print(items[0])

['A 35-year-old woman is evaluated for a long history of easy bruising. The peripheral smear shows only a few, large, young platelets, while other cell lines are normal. Marrow studies show increased megakaryocytes. Which of the following is the most likely diagnosis?', 'What is the most likely diagnosis in a 35-year-old woman with a history of easy bruising, a peripheral smear showing only a few large young platelets, and increased megakaryocytes in the marrow?']


In [31]:
original_questions = [item[0] for item in items]
for i, entry in enumerate(entries):
    if entry["question"] in original_questions:
        print("Match at index", i, "for question:", entry["question"])

### Fix question alignment

In [57]:
# train dev test
# question explanation
path = "cleaned_backup/dev_question.jsonl"

with jsonlines.open(path, "r") as f:
    entries = [entry for entry in f]

num_misaligned = 0
for i, entry in enumerate(entries):
    if entries[i]["question_upd"] == entries[i-1]["question_upd"]:
        num_misaligned += 1
        print(f"\nQuestion misalignment at index {i} and {i-1}, fixing...")
        if i < len(entries) - 1:
            entries[i]["question_upd"] = entries[i+1]["question_upd"]
            entries[i]["op_in_question_upd"] = entries[i+1]["op_in_question_upd"]
            entries[i]["comment_in_question_upd"] = entries[i+1]["comment_in_question_upd"]
            print("Original question:", entries[i]["question"])
            print("Updated question:", entries[i]["question_upd"])
        else:
            print(f"Cannot fix misalignment at index {i} because it's the last entry.")

print("Number of total entries:", len(entries))
print(f"Total number of misaligned questions: {num_misaligned}")

# with jsonlines.open(path, "w", flush=True) as f:
#     [f.write(entry) for entry in entries]

Number of total entries: 4183
Total number of misaligned questions: 0


### Add Strike-Though Numeration

In [33]:
def add_init_numeration() -> None:
    """
    Add an "i" field to each entry in the given split, starting from 1, and save the updated entries back to the file.
    """
    counter = 0

    for split in ["train", "dev", "test"]:
        with jsonlines.open(f"original/{split}.jsonl", "r") as f:
            entries = [entry for entry in f]

        for i, entry in enumerate(entries):
            counter += 1
            entries[i]["i"] = counter

        with jsonlines.open(f"original/{split}.jsonl", "w", flush=True) as f:
            [f.write(entry) for entry in entries]

In [34]:
add_init_numeration()

In [1]:
def add_numeration_with_reference(
        edit_folder: str,
        reference_folder: str,
        edit_flavour: str = None,
) -> None:
    """
    Add an "i" field to each entry in the given split, starting from 1, and save the updated entries back to the file.
    """
    for split in ["train", "dev", "test"]:
        print(f"Processing split: {split} with flavour: {edit_flavour}")

        ref_path = Path(f"{reference_folder}/{split}.jsonl")
        print("Reference path:", ref_path)
        with jsonlines.open(ref_path, "r") as f:
            ref_entries = [entry for entry in f]

        if edit_flavour:
            edit_path = Path(f"{edit_folder}/{split}_{edit_flavour}.jsonl")
        else:
            edit_path = Path(f"{edit_folder}/{split}.jsonl")
        print("Edit path:", edit_path)
        if not edit_path.exists():
            print(f"No edit file for: {split}, {edit_flavour}. Skipping.")
            continue

        with jsonlines.open(edit_path, "r") as f:
            edit_entries = [entry for entry in f]

        if "i" in edit_entries[0]:
            print(f"Entries in {edit_path} already have an 'i' field. Skipping.")
            continue

        i = 0
        for i, (ref_entry, edit_entry) in enumerate(zip(ref_entries, edit_entries)):
            assert ref_entry["question"] == edit_entry["question"], \
                (f"Question mismatch between reference and source at index {ref_entry['i']} ({i})\n"
                 f"Reference question: {ref_entry['question']}\n"
                 f"Edit question: {edit_entry['question']}")
            if "i" in edit_entry:
                raise ValueError(f"Entry at index {i} in {edit_folder}/{split}_{edit_flavour}.jsonl already has an 'i' field.")
            edit_entries[i]["i"] = ref_entry["i"]

        print("Finished at index", i, "out of", len(ref_entries)-1)

        with jsonlines.open(edit_path, "w", flush=True) as f:
            for entry in edit_entries:
                f.write(entry)

In [80]:
for flavour in ["question", "explanation"]:
    add_numeration_with_reference(
        edit_flavour=flavour,
        edit_folder="cleaned",
        reference_folder="original",
    )
    add_numeration_with_reference(
        edit_flavour=f"{flavour}_backup",
        edit_folder="cleaned",
        reference_folder="original",
    )
add_numeration_with_reference(
    edit_flavour="question_no_diff_check",
    edit_folder="cleaned",
    reference_folder="original",
)
add_numeration_with_reference(
    edit_flavour="question_till_50712",
    edit_folder="cleaned",
    reference_folder="original",
)

Processing split: train with flavour: question
Reference path: original/train.jsonl
Edit path: cleaned_no_numbering/train_question.jsonl


AssertionError: Question mismatch between reference and source at index 62703 (62702)
Reference question: True about High roughage in the diet is
Edit question: A lady with 8 wks pregnancy presented with random blood glucose of 177mg/d1. The treatment is:

In [3]:
add_numeration_with_reference(
    edit_folder="original/clean_spaces_id",
    reference_folder="original",
)

Processing split: train with flavour: None
Reference path: original/train.jsonl
Edit path: original/clean_spaces_id/train.jsonl
Entries in original/clean_spaces_id/train.jsonl already have an 'i' field. Skipping.
Processing split: dev with flavour: None
Reference path: original/dev.jsonl
Edit path: original/clean_spaces_id/dev.jsonl
Entries in original/clean_spaces_id/dev.jsonl already have an 'i' field. Skipping.
Processing split: test with flavour: None
Reference path: original/test.jsonl
Edit path: original/clean_spaces_id/test.jsonl
Entries in original/clean_spaces_id/test.jsonl already have an 'i' field. Skipping.


In [4]:
add_numeration_with_reference(
    edit_folder="cleaned",
    reference_folder="original/clean_spaces_id",
    edit_flavour="explanation",
)

Processing split: train with flavour: explanation
Reference path: original/clean_spaces_id/train.jsonl
Edit path: cleaned/train_explanation.jsonl


AssertionError: Question mismatch between reference and source at index 1 (0)
Reference question: Chronic urethral obstruction due to benign prismatic hyperplasia can lead to the following change in kidney parenchyma
Edit question: Which vitamin is supplied from only animal source:

### Standardize test set (add missing "exp" field)

In [10]:
path = "original/test.jsonl"
with jsonlines.open(path, "r") as f:
    entries = []
    for entry in f:
        if "exp" not in entry:
            entry["exp"] = None
        entries.append(entry)

with jsonlines.open(path, "w", flush=True) as f:
    for entry in entries:
        f.write(entry)

### Check Effectiveness of Context Classification

In [8]:
with jsonlines.open("cleaned/dev_explanation.jsonl", "r") as f:
    entries = [entry for entry in f]

num_exp = sum(1 for entry in entries if entry["exp_to_edit"] is False)
print(f"Number of entries classified as already good: {num_exp}/{len(entries)} ({num_exp/len(entries)*100:.2f}%)")

Number of entries classified as already good: 190/4183 (4.54%)


In [40]:
with jsonlines.open("cleaned/train_explanation.jsonl", "r") as f:
    entries = [entry for entry in f]

num_exp = sum(1 for entry in entries if entry["exp_to_edit"] is False)
print(f"Number of entries classified as already good: {num_exp}/{len(entries)} ({num_exp/len(entries)*100:.2f}%)")
# Number of entries classified as already good: 52282/131752 (39.68%)
# Number of entries classified as already good: 52282/131784 (39.67%)

Number of entries classified as already good: 52282/131784 (39.67%)


### Fix entry mismatch
(forgot to add entries with `exp_to_edit=None` for the ones where explanation is too long to process)

In [10]:
entries[-1]

{'question': 'Commonest site of feilization is :',
 'exp': 'Ampulla',
 'cop': 2,
 'opa': 'lsthmic',
 'opb': 'Ampulla',
 'opc': 'lnfundibulum',
 'opd': 'lnterstitial',
 'subject_name': 'Gynaecology & Obstetrics',
 'topic_name': None,
 'id': 'c8951b55-c9de-4c00-a5d2-eb1a118c96a7',
 'choice_type': 'single',
 'i': 131784,
 'exp_to_edit': False}

In [36]:
counter = 0
with jsonlines.open("cleaned/train_explanation.jsonl", "r") as f:
    cleaned_entries = [entry for entry in f]

with jsonlines.open("original/clean_spaces_id/train.jsonl", "r") as f:
    original_entries = [entry for entry in f]

cleaned_entries_upd = []
fixed = 0
for i, (e_orig, e_clean) in enumerate(zip(original_entries, cleaned_entries), start=1):
    if len(cleaned_entries_upd) > 1:
        if cleaned_entries_upd[-1]["i"] != cleaned_entries_upd[-2]["i"] + 1:
            missing_e = original_entries[i-2+fixed]
            missing_e["exp_to_edit"] = None
            cleaned_entries_upd.insert(i-2+fixed, missing_e)
            assert cleaned_entries_upd[-1]["i"] == cleaned_entries_upd[-2]["i"] + 1, \
                f"Numeration error at index {i}: {cleaned_entries_upd[-2]['i']} followed by {cleaned_entries_upd[-1]['i']}, missing_entry i ={missing_e['i']}, fixed count: {fixed}"
            fixed += 1
    cleaned_entries_upd.append(e_clean)

for i in range(1, len(cleaned_entries_upd)-1):
    if cleaned_entries_upd[i]["i"] != cleaned_entries_upd[i-1]["i"] + 1:
        print(f"Numeration mismatch at index {i}: {cleaned_entries_upd[i-1]['i']} followed by {cleaned_entries_upd[i]['i']}")
        counter += 1

print("Total mismatches:", counter)

Total mismatches: 0


In [37]:
with jsonlines.open("cleaned/train_explanation.jsonl", "w", flush=True) as f:
    for entry in cleaned_entries_upd:
        f.write(entry)